# Hugging Face Transformer Models for Kazakhstan News Classification

This notebook trains and compares several transformer-based models using the same train/validation/test split from `kaggle_train_dataset.csv`.

Models included:
- `xlm-roberta-base`
- `bert-base-multilingual-cased`
- `distilbert-base-multilingual-cased`

Outputs:
- validation/test metrics for each model
- saved CSV metrics
- Plotly comparison charts
- confusion matrix for the best transformer model
- predictions on `kazakhstan_inference_dataset.csv`


In [ ]:
# Install required libraries
!pip install transformers datasets accelerate evaluate plotly -q


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn plotly


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 1. Load Kaggle training dataset

In [ ]:
df_kaggle = pd.read_csv(
    'kaggle_train_dataset.csv',
    engine='python',
    on_bad_lines='warn',
    encoding='utf-8',
    encoding_errors='replace'
)

print(df_kaggle.shape)
df_kaggle.head()

(16230, 12)


,title,text,url,source,tags,category,char_count,word_count,title_len,date,input_text,label
0,Тоқаев қазақстандықтарды 2022 жылы не күтіп тұ...,Президент Қасым-Жомарт Тоқаев қазақстандықтард...,https://kaz.tengrinews.kz/kazakhstan_news/toka...,kaggle,"['тоқаев қасым-жомарт', 'жаңа жыл']",politics,1378,165,58,NaN,тоқаев қазақстандықтарды 2022 жылы не күтіп тұ...,6
1,Астанадағы тегін 5G: министрлік 8 орынды атады,5G жиілік бойынша аукционда жеңіске жеткен ұял...,https://kaz.tengrinews.kz/kazakhstan_news/asta...,kaggle,['министрлік'],politics,1088,134,46,NaN,астанадағы тегін 5g: министрлік 8 орынды атады...,6
2,Тоқаевқа 5G-ге көшуге дайындық туралы есеп бер...,"Мемлекет басшысы цифрлық даму, инновациялар жә...",https://kaz.tengrinews.kz/kazakhstan_news/toka...,kaggle,['тоқаев қасым-жомарт'],politics,3415,385,50,NaN,тоқаевқа 5g-ге көшуге дайындық туралы есеп бер...,6
3,"Мусин қазақстандықтарды 4G, 5G мұнарасынан қор...","Цифрлық даму, инновациялар және аэроғарыш өнер...",https://kaz.tengrinews.kz/kazakhstan_news/musi...,kaggle,"['қазақстан', 'коронавирус', 'интернет']",health,1205,147,61,NaN,"мусин қазақстандықтарды 4g, 5g мұнарасынан қор...",3
4,Сарапшы 5G технологияның адам ағзасына қалай ә...,"""СберЗдоровье"" медициналық сервисінің директор...",https://kaz.tengrinews.kz/damu/sarapshyi-5g-te...,kaggle,"['зерттеу', 'технология']",education,1031,124,64,NaN,сарапшы 5g технологияның адам ағзасына қалай ә...,2


In [ ]:
# Check label/category mapping
label_map = (
    df_kaggle[['label', 'category']]
    .drop_duplicates()
    .sort_values('label')
    .set_index('label')['category']
    .to_dict()
)

label_names = [label_map[i] for i in sorted(label_map.keys())]
num_labels = int(df_kaggle['label'].max() + 1)

print('Number of labels:', num_labels)
print(label_map)

Number of labels: 9
{0: 'crime', 1: 'economy', 2: 'education', 3: 'health', 4: 'international', 5: 'other', 6: 'politics', 7: 'social', 8: 'sports'}


## 2. Train / validation / test split

We use stratified split to preserve class distribution.

In [ ]:
X = df_kaggle['input_text'].astype(str)
y = df_kaggle['label'].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

Train: (11361,) (11361,)
Validation: (2434,) (2434,)
Test: (2435,) (2435,)


In [ ]:
train_df = pd.DataFrame({
    'text': X_train.reset_index(drop=True),
    'label': y_train.reset_index(drop=True)
})

val_df = pd.DataFrame({
    'text': X_val.reset_index(drop=True),
    'label': y_val.reset_index(drop=True)
})

test_df = pd.DataFrame({
    'text': X_test.reset_index(drop=True),
    'label': y_test.reset_index(drop=True)
})

## 3. Metrics function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'f1': f1_score(labels, preds, average='weighted', zero_division=0)
    }

## 4. Helper functions for training and evaluation

In [ ]:
def make_hf_datasets(model_name, max_length=256):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(batch):
        return tokenizer(
            batch['text'],
            truncation=True,
            max_length=max_length
        )

    train_dataset = Dataset.from_pandas(train_df).map(tokenize_function, batched=True)
    val_dataset = Dataset.from_pandas(val_df).map(tokenize_function, batched=True)
    test_dataset = Dataset.from_pandas(test_df).map(tokenize_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    return tokenizer, data_collator, train_dataset, val_dataset, test_dataset

In [ ]:
def train_transformer_model(
    model_name,
    output_dir,
    learning_rate=2e-5,
    num_train_epochs=3,
    batch_size=16,
    max_length=256,
    weight_decay=0.01
):
    print('=' * 100)
    print(f'Training model: {model_name}')
    print('=' * 100)

    tokenizer, data_collator, train_dataset, val_dataset, test_dataset = make_hf_datasets(
        model_name=model_name,
        max_length=max_length
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to='none'
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()

    val_metrics = trainer.evaluate(val_dataset)
    test_metrics = trainer.evaluate(test_dataset)
    test_predictions = trainer.predict(test_dataset)
    y_test_pred = np.argmax(test_predictions.predictions, axis=-1)

    print('Validation metrics:')
    print(val_metrics)
    print('Test metrics:')
    print(test_metrics)
    print('Classification report:')
    print(classification_report(
        y_test,
        y_test_pred,
        labels=sorted(label_map.keys()),
        target_names=label_names,
        zero_division=0
    ))

    result = {
        'Model': model_name,
        'Accuracy': test_metrics['eval_accuracy'],
        'Precision': test_metrics['eval_precision'],
        'Recall': test_metrics['eval_recall'],
        'F1-score': test_metrics['eval_f1'],
        'Validation F1-score': val_metrics['eval_f1'],
        'learning_rate': learning_rate,
        'epochs': num_train_epochs,
        'batch_size': batch_size,
        'max_length': max_length
    }

    return {
        'model_name': model_name,
        'trainer': trainer,
        'tokenizer': tokenizer,
        'data_collator': data_collator,
        'test_dataset': test_dataset,
        'y_test_pred': y_test_pred,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'result': result
    }

## 5. Train multiple transformer models

Recommended first run: `epochs=3`, `max_length=256`, `batch_size=16`.

If training is too slow, use `max_length=128` and/or `epochs=2` for a quick experiment.

In [ ]:
MODEL_CONFIGS = [
    {
        'model_name': 'xlm-roberta-base',
        'output_dir': './results_xlm_roberta_base',
        'learning_rate': 1e-5,
        'num_train_epochs': 4,
        'batch_size': 16,
        'max_length': 256
    },
    {
        'model_name': 'bert-base-multilingual-cased',
        'output_dir': './results_mbert_cased',
        'learning_rate': 2e-5,
        'num_train_epochs': 3,
        'batch_size': 16,
        'max_length': 256
    },
    {
        'model_name': 'distilbert-base-multilingual-cased',
        'output_dir': './results_distilbert_multilingual',
        'learning_rate': 2e-5,
        'num_train_epochs': 3,
        'batch_size': 16,
        'max_length': 256
    }
]

In [ ]:
transformer_outputs = {}
transformer_results = []

for cfg in MODEL_CONFIGS:
    output = train_transformer_model(**cfg)
    transformer_outputs[cfg['model_name']] = output
    transformer_results.append(output['result'])

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

transformer_results_df = pd.DataFrame(transformer_results)
transformer_results_df = transformer_results_df.sort_values('F1-score', ascending=False)
transformer_results_df.to_csv('transformer_models_test_metrics.csv', index=False)
transformer_results_df

Training model: xlm-roberta-base


Map:   0%|          | 0/11361 [00:00<?, ? examples/s]

Map:   0%|          | 0/2434 [00:00<?, ? examples/s]

Map:   0%|          | 0/2435 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Validation metrics:
{'eval_loss': 0.7832399606704712, 'eval_accuracy': 0.7345932621199671, 'eval_precision': 0.72936228378536, 'eval_recall': 0.7345932621199671, 'eval_f1': 0.7306806866613789, 'eval_runtime': 5.0984, 'eval_samples_per_second': 477.404, 'eval_steps_per_second': 30.009, 'epoch': 3.0}
Test metrics:
{'eval_loss': 0.7808375954627991, 'eval_accuracy': 0.7453798767967146, 'eval_precision': 0.7413259655990209, 'eval_recall': 0.7453798767967146, 'eval_f1': 0.7421002145328645, 'eval_runtime': 4.9802, 'eval_samples_per_second': 488.94, 'eval_steps_per_second': 30.722, 'epoch': 3.0}
Classification report:
               precision    recall  f1-score   support

        crime       0.60      0.49      0.54       173
      economy       0.68      0.52      0.59        96
    education       0.61      0.65      0.63        79
       health       0.85      0.86      0.85       428
international       0.61      0.52      0.56       137
        other       0.76      0.79      0.77       

,Model,Accuracy,Precision,Recall,F1-score,Validation F1-score,learning_rate,epochs,batch_size,max_length
1,bert-base-multilingual-cased,0.759343,0.755866,0.759343,0.756434,0.731912,0.00002,3,16,256
2,distilbert-base-multilingual-cased,0.745380,0.741326,0.745380,0.742100,0.730681,0.00002,3,16,256
0,xlm-roberta-base,0.740452,0.737924,0.740452,0.736886,0.724105,0.00001,4,16,256


In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

all_reports = []

for model_name, output in transformer_outputs.items():

    y_pred = output["y_test_pred"]

    report_dict = classification_report(
        y_test,
        y_pred,
        labels=sorted(label_map.keys()),
        target_names=label_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()
    report_df["Model"] = model_name

    file_name = f"{model_name.replace('/', '_')}_classification_report.csv"
    report_df.to_csv(file_name)

    print(f"Saved: {file_name}")

    all_reports.append(report_df)

all_transformer_reports_df = pd.concat(all_reports)

all_transformer_reports_df.to_csv(
    "all_transformer_classification_reports.csv"
)

all_transformer_reports_df

Saved: xlm-roberta-base_classification_report.csv
Saved: bert-base-multilingual-cased_classification_report.csv
Saved: distilbert-base-multilingual-cased_classification_report.csv


,precision,recall,f1-score,support,Model
crime,0.566845,0.612717,0.588889,173.000000,xlm-roberta-base
economy,0.658537,0.562500,0.606742,96.000000,xlm-roberta-base
education,0.560000,0.354430,0.434109,79.000000,xlm-roberta-base
health,0.866822,0.866822,0.866822,428.000000,xlm-roberta-base
international,0.649485,0.459854,0.538462,137.000000,xlm-roberta-base
other,0.760081,0.769388,0.764706,980.000000,xlm-roberta-base
politics,0.693252,0.771331,0.730210,293.000000,xlm-roberta-base
social,0.367647,0.403226,0.384615,62.000000,xlm-roberta-base
sports,0.858537,0.941176,0.897959,187.000000,xlm-roberta-base
accuracy,0.740452,0.740452,0.740452,0.740452,xlm-roberta-base


## 6. Plotly metrics comparison

In [ ]:
metrics_long = transformer_results_df.melt(
    id_vars='Model',
    value_vars=['Accuracy', 'Precision', 'Recall', 'F1-score'],
    var_name='Metric',
    value_name='Score'
)

fig = px.bar(
    metrics_long,
    x='Model',
    y='Score',
    color='Metric',
    barmode='group',
    text_auto='.3f',
    title='Transformer Models Metrics Comparison'
)

fig.update_layout(
    yaxis_range=[0, 1],
    xaxis_title='Model',
    yaxis_title='Score',
    width=1100,
    height=600
)

fig.show()
fig.write_html("transformer_metrics_comparison.html")

In [ ]:
fig = px.bar(
    transformer_results_df,
    x='Model',
    y='F1-score',
    text_auto='.3f',
    title='Transformer Models F1-score Comparison'
)

fig.update_layout(
    yaxis_range=[0, 1],
    xaxis_title='Model',
    yaxis_title='Weighted F1-score',
    width=900,
    height=550
)

fig.show()
fig.write_html("transformer_f1_comparison.html")

## 7. Confusion matrix for the best transformer model

In [ ]:
best_transformer_name = transformer_results_df.iloc[0]['Model']
best_output = transformer_outputs[best_transformer_name]
y_test_pred_best = best_output['y_test_pred']

cm = confusion_matrix(
    y_test,
    y_test_pred_best,
    labels=sorted(label_map.keys())
)

fig = ff.create_annotated_heatmap(
    z=cm,
    x=label_names,
    y=label_names,
    colorscale='Blues',
    showscale=True
)

fig.update_layout(
    title=f'Confusion Matrix - {best_transformer_name}',
    xaxis_title='Predicted Category',
    yaxis_title='True Category',
    width=950,
    height=750
)

fig.show()
fig.write_html("confusion_matrix_best_model.html")

In [ ]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff
import os

os.makedirs("plotly_graphs", exist_ok=True)

all_labels = sorted(label_map.keys())

model_predictions = {
    "mbert": transformer_outputs["bert-base-multilingual-cased"]["y_test_pred"],
    "distilbert": transformer_outputs["distilbert-base-multilingual-cased"]["y_test_pred"]
}

for model_name, y_pred in model_predictions.items():

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=all_labels
    )

    fig = ff.create_annotated_heatmap(
        z=cm,
        x=label_names,
        y=label_names,
        colorscale="Blues",
        showscale=True
    )

    fig.update_layout(
        title=f"Confusion Matrix - {model_name}",
        xaxis_title="Predicted Category",
        yaxis_title="True Category",
        width=950,
        height=750
    )

    fig.show()

    html_name = f"confusion_matrix_{model_name}.html"

    fig.write_html(html_name)

    print(f"Saved: {html_name}")

Saved: confusion_matrix_mbert.html


Saved: confusion_matrix_distilbert.html


## 8. Class-wise F1-score for the best transformer model

In [ ]:
report = classification_report(
    y_test,
    y_test_pred_best,
    labels=sorted(label_map.keys()),
    target_names=label_names,
    output_dict=True,
    zero_division=0
)

class_f1_df = pd.DataFrame(report).T.reset_index().rename(columns={'index': 'Category'})
class_f1_df = class_f1_df[class_f1_df['Category'].isin(label_names)]

fig = px.bar(
    class_f1_df,
    x='Category',
    y='f1-score',
    text_auto='.3f',
    title=f'Class-wise F1-score - {best_transformer_name}'
)

fig.update_layout(
    yaxis_range=[0, 1],
    xaxis_title='Category',
    yaxis_title='F1-score',
    width=950,
    height=550
)

fig.show()

class_f1_df.to_csv('best_transformer_classwise_f1.csv', index=False)
class_f1_df
fig.write_html("class_wise_f1_score.html")

## 9. Predict on Kazakhstan inference dataset

The inference dataset has placeholder labels (`other`), so we do not calculate quantitative metrics on it. We only generate predictions and analyze predicted category distribution.

In [ ]:
inference_df = pd.read_csv(
    'kazakhstan_inference_dataset.csv',
    engine='python',
    on_bad_lines='warn',
    encoding='utf-8',
    encoding_errors='replace'
)

inference_texts = inference_df['input_text'].astype(str)
inference_hf_df = pd.DataFrame({'text': inference_texts})

best_trainer = best_output['trainer']
best_tokenizer = best_output['tokenizer']

inference_dataset = Dataset.from_pandas(inference_hf_df)

def tokenize_inference(batch):
    return best_tokenizer(
        batch['text'],
        truncation=True,
        max_length=int(best_output['result']['max_length'])
    )

inference_dataset = inference_dataset.map(tokenize_inference, batched=True)

inference_predictions = best_trainer.predict(inference_dataset)
inference_pred_labels = np.argmax(inference_predictions.predictions, axis=-1)

inference_df['predicted_label_transformer'] = inference_pred_labels
inference_df['predicted_category_transformer'] = inference_df['predicted_label_transformer'].map(label_map)

inference_df.to_csv('kazakhstan_inference_best_transformer_predictions.csv', index=False)

inference_df[['input_text', 'predicted_label_transformer', 'predicted_category_transformer']].head()

Parameter 'function'=<function tokenize_inference at 0x7cee4027a840> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/11513 [00:00<?, ? examples/s]

,input_text,predicted_label_transformer,predicted_category_transformer
0,"астана, алматы және шымкентте есірткі тұтынушы...",0,crime
1,“ешқандай таныс ықпал ете алмайды“ - вице-мини...,5,other
2,“сіз iphone сатып алдыңыз“: қазақстандықтарға ...,0,crime
3,астанада парламент палаталарының бірлескен оты...,6,politics
4,астана мен алматыда 14-16 мамырда ауа райы қан...,5,other


In [ ]:
pred_counts_df = (
    inference_df['predicted_category_transformer']
    .value_counts()
    .reset_index()
)
pred_counts_df.columns = ['Predicted Category', 'Count']

fig = px.bar(
    pred_counts_df,
    x='Predicted Category',
    y='Count',
    text_auto=True,
    title=f'Predicted Category Distribution on Kazakhstan Inference Dataset - {best_transformer_name}'
)

fig.update_layout(
    xaxis_title='Predicted Category',
    yaxis_title='Number of Articles',
    width=950,
    height=550
)

fig.show()

pred_counts_df
fig.write_html("kz_inference_predicted.html")

## Error Analysis

In [ ]:
# Build error dataframe using best model predictions
error_df = pd.DataFrame({
    'text': X_test.reset_index(drop=True),
    'true_label': y_test.reset_index(drop=True),
    'pred_label': y_test_pred_best
})

error_df['true_category'] = error_df['true_label'].map(label_map)
error_df['pred_category'] = error_df['pred_label'].map(label_map)
error_df['is_error'] = error_df['true_label'] != error_df['pred_label']
error_df['text_length'] = error_df['text'].str.len()

total = len(error_df)
n_errors = error_df['is_error'].sum()
print(f'Total test samples : {total}')
print(f'Misclassified      : {n_errors} ({n_errors/total*100:.1f}%)')
print(f'Correctly classified: {total - n_errors} ({(total-n_errors)/total*100:.1f}%)')

Total test samples : 2435
Misclassified      : 632 (26.0%)
Correctly classified: 1803 (74.0%)


In [ ]:
#Top confused class pairs
errors_only = error_df[error_df['is_error']].copy()

confusion_pairs = (
    errors_only
    .groupby(['true_category', 'pred_category'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(10)
)
confusion_pairs['pair'] = confusion_pairs['true_category'] + ' → ' + confusion_pairs['pred_category']

fig = px.bar(
    confusion_pairs,
    x='count',
    y='pair',
    orientation='h',
    text_auto=True,
    title=f'Top 10 Most Confused Class Pairs — {best_transformer_name}',
    labels={'count': 'Number of Errors', 'pair': 'True → Predicted'}
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)
fig.show()
fig.write_html('error_analysis_confused_pairs.html')

print('\nTop confused pairs:')
print(confusion_pairs[['true_category', 'pred_category', 'count']].to_string(index=False))


Top confused pairs:
true_category pred_category  count
        other         crime     58
        other      politics     56
        crime         other     55
     politics         other     37
international         other     36
       health         other     30
    education         other     27
      economy         other     26
        other        health     25
        other        sports     25


In [ ]:
#Per-class error rate
class_stats = error_df.groupby('true_category').agg(
    total=('is_error', 'count'),
    errors=('is_error', 'sum')
).reset_index()
class_stats['error_rate'] = class_stats['errors'] / class_stats['total']
class_stats = class_stats.sort_values('error_rate', ascending=False)

fig = px.bar(
    class_stats,
    x='true_category',
    y='error_rate',
    text_auto='.2%',
    title='Error Rate per Class',
    labels={'true_category': 'Category', 'error_rate': 'Error Rate'}
)
fig.update_layout(yaxis_tickformat='.0%', height=450)
fig.show()
fig.write_html('error_analysis_per_class_error_rate.html')

print(class_stats.to_string(index=False))

true_category  total  errors  error_rate
    education     79      51    0.645570
       social     62      37    0.596774
international    137      74    0.540146
      economy     96      42    0.437500
        crime    173      67    0.387283
        other    980     226    0.230612
     politics    293      67    0.228669
       health    428      57    0.133178
       sports    187      11    0.058824


In [ ]:
#Text length vs error rate
error_df['length_bin'] = pd.cut(
    error_df['text_length'],
    bins=[0, 100, 300, 600, 1000, 99999],
    labels=['<100', '100-300', '300-600', '600-1000', '>1000']
)

length_stats = error_df.groupby('length_bin', observed=True).agg(
    total=('is_error', 'count'),
    errors=('is_error', 'sum')
).reset_index()
length_stats['error_rate'] = length_stats['errors'] / length_stats['total']

fig = px.bar(
    length_stats,
    x='length_bin',
    y='error_rate',
    text_auto='.2%',
    title='Error Rate by Text Length',
    labels={'length_bin': 'Text Length (characters)', 'error_rate': 'Error Rate'}
)
fig.update_layout(yaxis_tickformat='.0%', height=450)
fig.show()
fig.write_html('error_analysis_text_length.html')

print(length_stats.to_string(index=False))

length_bin  total  errors  error_rate
   300-600     17       1    0.058824
  600-1000    424      88    0.207547
     >1000   1994     543    0.272317


In [ ]:
#Save full error table to CSV
errors_only[['text', 'true_category', 'pred_category', 'text_length']].to_csv(
    'error_analysis_misclassified.csv', index=False
)
print(f'Saved {len(errors_only)} misclassified examples to error_analysis_misclassified.csv')

Saved 632 misclassified examples to error_analysis_misclassified.csv


## Key findings:

### **1. Class `other` is the primary source of confusion — it overlaps heavily**
   with crime, politics, health, and sports. This suggests the category
   is too broad and semantically ambiguous.

### **2. Education (64.6% error rate) and Social (59.7%) are the hardest classes.**
   These topics share vocabulary with politics and economy, making them
   difficult to distinguish without deeper semantic understanding.

### **3. Sports (5.9%) and Health (13.3%) are easiest — they have distinct,**
   domain-specific vocabulary that transformers handle well.

### **4. Longer texts have higher error rates, likely because long articles**
   tend to belong to ambiguous categories (other, politics, economy)
   rather than because length itself causes errors.

